# Laguna XS.2 Expert Atlas — Kaggle T4×2

This notebook implements the **selection stage** for precise MoE surgery on `poolside/Laguna-XS.2-INT4`.

Pipeline:

`9,984 layer-expert blocks → one routing sweep → selective layers → selective experts → fixed-routing causal ablation → ranked surgery candidates`

XS.2 has 40 layers, with 39 sparse MoE layers, 256 routed experts, top-8 routing, hidden size 2048, expert width 512, and roughly 33B total / 3B active parameters.

**Kaggle:** choose **GPU T4 ×2** and enable Internet for the first model download.

Why INT4: the BF16 checkpoint is too large for T4×2. During atlas work we use `use_cache=False`, so the checkpoint's FP8 KV cache is not used.

Official sources:
- https://huggingface.co/poolside/Laguna-XS.2
- https://huggingface.co/poolside/Laguna-XS.2-INT4
- https://huggingface.co/poolside/Laguna-XS.2/blob/main/config.json
- https://github.com/huggingface/transformers/blob/main/src/transformers/models/laguna/modeling_laguna.py

## 1) Install

In [ ]:
!pip -q install -U "transformers>=5.7.0" "accelerate>=1.10.0" compressed-tensors safetensors huggingface_hub pandas pyarrow tqdm matplotlib

## 2) Hardware check

In [ ]:
import os, sys, json, math, time, gc, random, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch

print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator.'
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p=torch.cuda.get_device_properties(i)
    print(i, p.name, f'{p.total_memory/2**30:.2f} GiB', f'cc {p.major}.{p.minor}')
if torch.cuda.device_count() < 2:
    print('WARNING: tuned for Kaggle T4×2; one T4 is unlikely to fit XS.2 INT4 with enough headroom.')
os.system('nvidia-smi')

## 3) Configuration

In [ ]:
MODEL_ID = 'poolside/Laguna-XS.2-INT4'
TARGET_CATEGORY = 'frontend'   # change: python/java/cpp/shell/sql/algorithms/math/frontend
MAX_PROBE_TOKENS = 192
TOP_LAYERS = 8
EXPERTS_PER_LAYER = 5
CAUSAL_TARGET_N = 4            # smoke test; raise to 20-100+ for real confidence
CAUSAL_CONTROL_N = 4
RENORMALIZE_AFTER_ABLATION = False
SEED = 42
OUT = Path('/kaggle/working/laguna_xs2_expert_atlas')
OUT.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 4) Load XS.2 INT4 across both T4s

Transformers 5.7+ supports Laguna. `device_map="balanced"` shards layers across the two GPUs. T4 uses FP16 activations. We force SDPA and leave VRAM headroom for router tensors and causal scoring.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
max_memory = {i:'14GiB' for i in range(torch.cuda.device_count())}
max_memory['cpu']='8GiB'

try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        dtype=torch.float16,
        device_map='balanced',
        max_memory=max_memory,
        low_cpu_mem_usage=True,
        offload_folder=str(OUT/'offload'),
        offload_state_dict=True,
        attn_implementation='sdpa',
    )
except Exception as e:
    print(type(e).__name__, e)
    print('\nDo not switch to GGUF for this atlas: we need internal router tensors. Verify T4×2, Internet, and Transformers >=5.7.')
    raise

model.eval(); model.config.use_cache=False
gc.collect(); torch.cuda.empty_cache()
print('Loaded', MODEL_ID)
print(model.hf_device_map)

## 5) Verify MoE structure

In [ ]:
cfg=model.config
sparse_layers=[i for i,l in enumerate(model.model.layers) if hasattr(l.mlp,'gate') and hasattr(l.mlp,'experts')]
print('layers', cfg.num_hidden_layers, 'sparse', len(sparse_layers))
print('experts', cfg.num_experts, 'top-k', cfg.num_experts_per_tok)
print('hidden', cfg.hidden_size, 'expert width', cfg.moe_intermediate_size)
assert len(sparse_layers)==39 and cfg.num_experts==256 and cfg.num_experts_per_tok==8
params_per_expert=3*cfg.hidden_size*cfg.moe_intermediate_size
print('approx params/expert:', f'{params_per_expert/1e6:.3f}M')
print('one expert per sparse layer:', f'{params_per_expert*len(sparse_layers)/1e6:.1f}M')

## 6) Multi-domain routing probes

These are a **search index**, not a benchmark. For high precision, expand each category with real examples from the target distribution.

In [ ]:
PROBES={
'frontend':[
'Fix a React flex layout whose children overflow horizontally on mobile.',
'A button shifts when a loading spinner appears; diagnose the CSS layout cause.',
'Explain a z-index bug caused by a stacking context in a modal.',
'Debug a Next.js hydration mismatch caused by browser-only state.',
'Correct padding, line-height, border radius, and flex alignment to match a UI screenshot.',
'Explain why width:100vw can create a horizontal scrollbar.',
'Implement an accessible dropdown with keyboard navigation and focus handling.',
'Explain why a CSS transform can affect fixed-position descendants.'
],
'python':[
'Write a Python retry decorator with exponential backoff and jitter.',
'Debug a Python generator that stops one item too early.',
'Explain dataclasses versus NamedTuple in Python.',
'Fix an async Python function that calls blocking requests.get.',
'Stream a huge JSONL file without loading it all into memory.',
'Explain Python descriptor lookup order.',
'Implement a small LRU cache with OrderedDict.',
'Explain the mutable-default-argument bug.'
],
'java':[
'Implement Dijkstra in Java using PriorityQueue and adjacency lists.',
'Explain why equals without hashCode breaks HashMap behavior.',
'Debug ConcurrentModificationException in an enhanced for-loop.',
'Implement a thread-safe lazy singleton in modern Java.',
'Fix a Comparator that subtracts integers.',
'Write BFS over a grid in Java.',
'Explain checked versus unchecked exceptions.',
'Combine two operations with CompletableFuture.'
],
'cpp':[
'Explain move construction versus copy construction in modern C++.',
'Find a use-after-free caused by vector reallocation.',
'Implement Dijkstra in C++ using priority_queue.',
'Explain RAII and unique_ptr.',
'Debug returning a reference to a local variable.',
'Fix iterator invalidation after vector::erase.',
'Explain false sharing in multithreaded C++.',
'Implement a bounded producer-consumer queue.'
],
'shell':[
'Find the ten largest files recursively while ignoring permission errors.',
'Explain stdout versus stderr redirection.',
'Debug a pipeline whose final exit code hides an earlier failure.',
'Write a safe bash loop over filenames containing spaces.',
'Use find and xargs safely with unusual filenames.',
'Inspect which process listens on TCP port 8080.',
'Create a tar.gz excluding node_modules and .git.',
'Write a trap that cleans up a temporary directory.'
],
'sql':[
'Return the second-highest salary without assuming uniqueness.',
'Explain how LEFT JOIN can become INNER JOIN after a WHERE filter.',
'Use a window function for a running total per customer.',
'Find duplicate emails and their counts.',
'Write latest-order-per-customer using ROW_NUMBER.',
'Explain an index on (user_id, created_at).',
'Fix double-counted revenue after joining two one-to-many tables.',
'Write a recursive CTE for a parent-child hierarchy.'
],
'algorithms':[
'Derive the one-dimensional coin-change dynamic programming recurrence.',
'Explain why BFS gives shortest paths in an unweighted graph.',
'Compare O(n^2) and O(n log n) longest-increasing-subsequence methods.',
'Use a monotonic stack for next greater element.',
'Derive the two-pointer trapping-rain-water algorithm.',
'Explain union-find with path compression and union by rank.',
'Solve minimum path sum using bottom-up DP.',
'Explain binary search on the answer.'
],
'math':[
'Prove the sum of the first n odd integers equals n squared.',
'Differentiate x^x for positive x.',
'Solve a two-equation linear system by elimination.',
'Explain the geometric meaning of a dot product.',
'Compute expected heads in ten fair coin flips.',
'Use Bayes theorem with a noisy diagnostic test.',
'Explain why the harmonic series diverges.',
'Derive the quadratic formula by completing the square.'
],
'general_reasoning':[
'Identify the actual critical path before rescheduling a delayed project.',
'Compare a high-upside risky plan with a lower-upside robust plan.',
'List causal explanations when a measurement changes after replacing the instrument.',
'Distinguish correlation from causation in observational data.',
'Which statistics reveal rare severe latency spikes?',
'Break a vague objective into constraints, assumptions, and experiments.',
'Explain how optimizing a proxy metric can hurt the true objective.',
'Design a falsifiable experiment for two competing explanations.'
]}
rows=[]; pid=0
for cat,ps in PROBES.items():
    for p in ps:
        rows.append({'probe_id':pid,'category':cat,'prompt':p}); pid+=1
probe_df=pd.DataFrame(rows)
print(probe_df.groupby('category').size(), '\nTotal:',len(probe_df))

## 7) Exact router reconstruction

Laguna uses `sigmoid(router_logits)`, adds each layer's fixed correction bias **only for top-k selection**, then normalizes the selected experts' unbiased sigmoid scores. We reconstruct that exactly.

In [ ]:
from tqdm.auto import tqdm

def input_device(): return model.model.embed_tokens.weight.device

def encode_probe(prompt):
    txt=tokenizer.apply_chat_template([{'role':'user','content':prompt}], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    b=tokenizer(txt, return_tensors='pt', truncation=True, max_length=MAX_PROBE_TOKENS, add_special_tokens=False)
    return {k:v.to(input_device()) for k,v in b.items()}

@torch.inference_mode()
def route_prompt(prompt):
    b=encode_probe(prompt); ntok=int(b['attention_mask'].sum())
    out=model(**b, use_cache=False, output_router_logits=True, logits_to_keep=1, return_dict=True)
    rlog=out.router_logits
    assert rlog is not None and len(rlog)==len(sparse_layers)
    rows=[]
    for pos,layer_idx in enumerate(sparse_layers):
        logits=rlog[pos].reshape(-1,cfg.num_experts)[:ntok].float()
        bias=model.model.layers[layer_idx].mlp.gate.e_score_correction_bias.to(logits.device, dtype=torch.float32)
        sig=torch.sigmoid(logits)
        selected=torch.topk(sig+bias, k=cfg.num_experts_per_tok, dim=-1).indices
        raw=sig.gather(-1,selected)
        weights=raw/raw.sum(-1,keepdim=True).clamp_min(1e-12)
        flat_e=selected.reshape(-1); flat_w=weights.reshape(-1)
        counts=torch.bincount(flat_e,minlength=cfg.num_experts).float()
        wsum=torch.zeros(cfg.num_experts,device=logits.device); wsum.scatter_add_(0,flat_e,flat_w)
        counts=counts.cpu().numpy(); wsum=wsum.cpu().numpy()
        for e in np.flatnonzero(counts>0):
            c=float(counts[e]); w=float(wsum[e])
            rows.append({'layer':layer_idx,'expert':int(e),'token_count':ntok,'selected_count':c,
                         'selected_rate':c/ntok,'routing_mass':w/ntok,'mean_weight_when_selected':w/max(c,1)})
    del out,rlog
    return rows

## 8) One-pass routing profile

In [ ]:
profile_rows=[]
for r in tqdm(probe_df.itertuples(index=False), total=len(probe_df), desc='routing'):
    for x in route_prompt(r.prompt):
        x['probe_id']=int(r.probe_id); x['category']=r.category; profile_rows.append(x)
profile=pd.DataFrame(profile_rows)
profile.to_parquet(OUT/'probe_routing_profile.parquet',index=False)
print('profile rows:',len(profile))
profile.head()

## 9) Preliminary expert atlas + automatic role hypotheses

In [ ]:
activity=(profile.groupby(['category','layer','expert'],as_index=False)
          .agg(routing_mass=('routing_mass','mean'),selected_rate=('selected_rate','mean'),prompts_active=('probe_id','nunique')))
full=pd.MultiIndex.from_product([sorted(PROBES),sparse_layers,range(cfg.num_experts)],names=['category','layer','expert'])
activity=activity.set_index(['category','layer','expert']).reindex(full).fillna(0).reset_index()

t=activity[activity.category==TARGET_CATEGORY][['layer','expert','routing_mass','selected_rate']].rename(columns={'routing_mass':'target_mass','selected_rate':'target_selected_rate'})
b=(activity[activity.category!=TARGET_CATEGORY].groupby(['layer','expert'],as_index=False)
   .agg(background_mass=('routing_mass','mean'),background_selected_rate=('selected_rate','mean')))
atlas=t.merge(b,on=['layer','expert']); eps=1e-8
atlas['lift']=atlas.target_mass-atlas.background_mass
atlas['ratio']=(atlas.target_mass+eps)/(atlas.background_mass+eps)
atlas['selectivity']=atlas['lift']/(atlas.target_mass+atlas.background_mass+eps)
atlas['routing_shortlist_score']=atlas['lift'].clip(lower=0)*np.sqrt(atlas.target_mass.clip(lower=0)+eps)*atlas.selectivity.clip(lower=0)
atlas=atlas.sort_values('routing_shortlist_score',ascending=False).reset_index(drop=True)

pivot=activity.pivot_table(index=['layer','expert'],columns='category',values='routing_mass',fill_value=0)
def role(layer,expert,n=3):
    s=pivot.loc[(layer,expert)].sort_values(ascending=False).head(n)
    return ' > '.join(f'{k} ({v:.4g})' for k,v in s.items())
atlas['role_hypothesis']=[role(int(r.layer),int(r.expert)) for r in atlas.itertuples(index=False)]
atlas.to_csv(OUT/f'preliminary_atlas_{TARGET_CATEGORY}.csv',index=False)
atlas.head(30)

## 10) Shortlist layers without brute-forcing all 39

In [ ]:
def top_positive_sum(s,k=8):
    x=np.sort(np.clip(np.asarray(s,float),0,None))[::-1]
    return float(x[:k].sum())
layer_routing=(atlas.groupby('layer').agg(layer_routing_score=('routing_shortlist_score',lambda s:top_positive_sum(s,8)),best_expert_score=('routing_shortlist_score','max')).reset_index().sort_values('layer_routing_score',ascending=False))
candidate_layers=layer_routing.head(TOP_LAYERS).layer.astype(int).tolist()
print('candidate layers:',candidate_layers)
layer_routing.head(15)

## 11) Causal target/control set

The bundled set is only a **frontend smoke test**. For any other target, replace it with target-specific `(prompt, reference)` pairs. For real precision use matched target/control examples and at least 20–100+ examples per side.

In [ ]:
CAUSAL=[
{'category':'frontend','prompt':'Return only the CSS declaration that lets a flex item shrink below its content width.','reference':'min-width: 0;'},
{'category':'frontend','prompt':'Return only the CSS declaration that creates a flex formatting context.','reference':'display: flex;'},
{'category':'frontend','prompt':'Return only the CSS declaration that centers flex items on the main axis.','reference':'justify-content: center;'},
{'category':'frontend','prompt':'In React, return only the hook name used for local component state.','reference':'useState'},
{'category':'frontend','prompt':'Return only the CSS property name that controls stacking order.','reference':'z-index'},
{'category':'frontend','prompt':'Return only the CSS property name used to hide horizontal overflow.','reference':'overflow-x'},
{'category':'python','prompt':'Return only the Python keyword that yields a generator value.','reference':'yield'},
{'category':'python','prompt':'Return only the immutable built-in ordered-sequence type in Python.','reference':'tuple'},
{'category':'algorithms','prompt':'Return only the traversal acronym for shortest paths in an unweighted graph.','reference':'BFS'},
{'category':'sql','prompt':'Return only the SQL keyword that removes duplicate SELECT rows.','reference':'DISTINCT'},
{'category':'java','prompt':'Return only the Java keyword used to inherit from a class.','reference':'extends'},
{'category':'cpp','prompt':'Return only the C++ smart pointer type expressing exclusive ownership.','reference':'std::unique_ptr'}]
causal_df=pd.DataFrame(CAUSAL)
if TARGET_CATEGORY!='frontend':
    print('WARNING: replace CAUSAL with target-specific prompt/reference pairs before trusting the causal ranking.')

## 12) Teacher-forced reference NLL

In [ ]:
def prefix_ids(prompt):
    txt=tokenizer.apply_chat_template([{'role':'user','content':prompt}], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    return tokenizer(txt,add_special_tokens=False,return_tensors='pt')['input_ids'][0]

def ref_ids(ref): return tokenizer(ref,add_special_tokens=False,return_tensors='pt')['input_ids'][0][:64]

@torch.inference_mode()
def reference_nll(prompt,ref):
    p=prefix_ids(prompt); r=ref_ids(ref); ids=torch.cat([p,r]).unsqueeze(0); labels=ids.clone(); labels[:,:p.numel()]=-100
    dev=input_device(); ids=ids.to(dev); labels=labels.to(dev); mask=torch.ones_like(ids)
    return float(model(input_ids=ids,attention_mask=mask,labels=labels,use_cache=False,output_router_logits=False,return_dict=True).loss.detach().cpu())

def eval_rows(df): return np.asarray([reference_nll(r.prompt,r.reference) for r in df.itertuples(index=False)],dtype=float)

target_eval=causal_df[causal_df.category==TARGET_CATEGORY].head(CAUSAL_TARGET_N)
control_eval=causal_df[causal_df.category!=TARGET_CATEGORY].head(CAUSAL_CONTROL_N)
assert len(target_eval)>0 and len(control_eval)>0, 'Need target and control causal examples.'
baseline_target=eval_rows(target_eval); baseline_control=eval_rows(control_eval)
print('baseline target',baseline_target,'mean',baseline_target.mean())
print('baseline control',baseline_control,'mean',baseline_control.mean())

## 13) Fixed-routing causal interventions

We keep the original top-8 expert IDs fixed. For an ablated expert we set only its selected routing weight to zero, preventing the router from silently substituting the 9th-ranked expert. Optional renormalization can be tested later.

In [ ]:
import types
from contextlib import contextmanager

def gate_for(layer): return model.model.layers[layer].mlp.gate

@contextmanager
def ablate_layer(layer):
    g=gate_for(layer); orig=g.forward
    def patched(self,h):
        logits,w,sel=orig(h); return logits,torch.zeros_like(w),sel
    g.forward=types.MethodType(patched,g)
    try: yield
    finally: g.forward=orig

@contextmanager
def ablate_experts(layer,expert_ids,renorm=False):
    ids={int(x) for x in expert_ids}; g=gate_for(layer); orig=g.forward
    def patched(self,h):
        logits,w,sel=orig(h); keep=torch.ones_like(w,dtype=torch.bool)
        for e in ids: keep &= (sel!=e)
        nw=w*keep.to(w.dtype)
        if renorm:
            d=nw.sum(-1,keepdim=True); nw=torch.where(d>0,nw/d.clamp_min(1e-12),nw)
        return logits,nw,sel
    g.forward=types.MethodType(patched,g)
    try: yield
    finally: g.forward=orig

## 14) Causal layer screen — only shortlisted layers

In [ ]:
rows=[]
for layer in tqdm(candidate_layers,desc='layer ablation'):
    with ablate_layer(layer): t=eval_rows(target_eval); c=eval_rows(control_eval)
    td=float((t-baseline_target).mean()); cd=float((c-baseline_control).mean())
    rows.append({'layer':layer,'target_delta_nll':td,'control_delta_nll':cd,'target_specific_delta':td-cd})
layer_causal=pd.DataFrame(rows).sort_values('target_specific_delta',ascending=False)
layer_causal.to_csv(OUT/f'layer_causal_{TARGET_CATEGORY}.csv',index=False)
layer_causal

## 15) Expert shortlist inside the screened layers

In [ ]:
lmap=layer_causal.set_index('layer').target_specific_delta.to_dict(); parts=[]
for layer in candidate_layers:
    q=atlas[atlas.layer==layer].copy(); q['layer_causal_score']=max(0,float(lmap.get(layer,0)))
    q['combined_precausal_score']=q.routing_shortlist_score*(1+q.layer_causal_score)
    parts.append(q.sort_values('combined_precausal_score',ascending=False).head(EXPERTS_PER_LAYER))
shortlist=pd.concat(parts,ignore_index=True).sort_values('combined_precausal_score',ascending=False).reset_index(drop=True)
print('experts to test:',len(shortlist))
shortlist[['layer','expert','role_hypothesis','target_mass','background_mass','selectivity','layer_causal_score','combined_precausal_score']]

## 16) Exact expert causal ablations

In [ ]:
rows=[]
for r in tqdm(shortlist.itertuples(index=False),total=len(shortlist),desc='expert ablation'):
    with ablate_experts(int(r.layer),[int(r.expert)],RENORMALIZE_AFTER_ABLATION): t=eval_rows(target_eval); c=eval_rows(control_eval)
    td=float((t-baseline_target).mean()); cd=float((c-baseline_control).mean())
    rows.append({'layer':int(r.layer),'expert':int(r.expert),'target_delta_nll':td,'control_delta_nll':cd,
                 'causal_selectivity':td-cd,'target_preserving_score':td-0.75*max(cd,0)})
causal=pd.DataFrame(rows)
final=shortlist.merge(causal,on=['layer','expert'])
final['surgery_score']=final.combined_precausal_score.clip(lower=0)*final.target_preserving_score.clip(lower=0)
final=final.sort_values(['surgery_score','causal_selectivity'],ascending=False).reset_index(drop=True)
final.to_csv(OUT/f'surgery_candidates_{TARGET_CATEGORY}.csv',index=False)
final[['layer','expert','role_hypothesis','target_mass','background_mass','target_delta_nll','control_delta_nll','causal_selectivity','surgery_score']].head(30)

## 17) Interpret candidates with their strongest target probes

In [ ]:
def top_examples(layer,expert,n=3):
    q=profile[(profile.layer==layer)&(profile.expert==expert)&(profile.category==TARGET_CATEGORY)].sort_values('routing_mass',ascending=False).head(n)
    return q.merge(probe_df,on=['probe_id','category'])[['probe_id','routing_mass','prompt']].to_dict('records')
records=[]
for r in final.itertuples(index=False):
    d=r._asdict(); d['top_target_probes']=top_examples(int(r.layer),int(r.expert)); records.append(d)
with open(OUT/f'surgery_candidates_{TARGET_CATEGORY}.json','w') as f: json.dump(records,f,indent=2)
for d in records[:5]:
    print('\n'+'='*72); print('L',d['layer'],'E',d['expert'],'score',d['surgery_score']); print('role:',d['role_hypothesis'])
    print('target ΔNLL',d['target_delta_nll'],'control ΔNLL',d['control_delta_nll'])
    for x in d['top_target_probes']: print(' -',x['prompt'])

## 18) Optional pair/coalition test

A capability may be distributed across an expert circuit rather than one expert. Enable this only after reviewing single-expert results.

In [ ]:
from itertools import combinations
RUN_COALITIONS=False
if RUN_COALITIONS:
    rows=[]
    for layer,g in final.groupby('layer'):
        eids=g.head(3).expert.astype(int).tolist()
        for pair in combinations(eids,2):
            with ablate_experts(int(layer),pair,RENORMALIZE_AFTER_ABLATION): t=eval_rows(target_eval); c=eval_rows(control_eval)
            td=float((t-baseline_target).mean()); cd=float((c-baseline_control).mean())
            rows.append({'layer':int(layer),'experts':list(pair),'target_delta_nll':td,'control_delta_nll':cd,'causal_selectivity':td-cd})
    coal=pd.DataFrame(rows).sort_values('causal_selectivity',ascending=False)
    coal.to_json(OUT/f'coalitions_{TARGET_CATEGORY}.json',orient='records',indent=2)
    display(coal.head(20))
else: print('Skipped; set RUN_COALITIONS=True when ready.')

## 19) Heatmap + export ZIP

In [ ]:
import matplotlib.pyplot as plt
pairs=atlas.head(64)[['layer','expert']].copy(); pairs['label']=[f'L{l}:E{e}' for l,e in zip(pairs.layer,pairs.expert)]
cats=sorted(PROBES); heat=np.array([[float(pivot.loc[(int(r.layer),int(r.expert))].get(cat,0)) for cat in cats] for r in pairs.itertuples(index=False)])
fig,ax=plt.subplots(figsize=(12,14)); im=ax.imshow(heat,aspect='auto'); ax.set_xticks(range(len(cats))); ax.set_xticklabels(cats,rotation=45,ha='right'); ax.set_yticks(range(len(pairs))); ax.set_yticklabels(pairs.label,fontsize=8); ax.set_title(f'XS.2 routing mass — {TARGET_CATEGORY} shortlist'); fig.colorbar(im,ax=ax,label='routing mass/token'); plt.tight_layout(); fig.savefig(OUT/f'routing_heatmap_{TARGET_CATEGORY}.png',dpi=160,bbox_inches='tight'); plt.show()
manifest={'model_id':MODEL_ID,'target_category':TARGET_CATEGORY,'sparse_layers':sparse_layers,'num_experts':cfg.num_experts,'top_k':cfg.num_experts_per_tok,'candidate_layers':candidate_layers,'experts_causally_tested':len(shortlist),'renormalized_ablation':RENORMALIZE_AFTER_ABLATION}
with open(OUT/'manifest.json','w') as f: json.dump(manifest,f,indent=2)
zip_path=shutil.make_archive('/kaggle/working/laguna_xs2_expert_atlas_results','zip',root_dir=OUT)
print('RESULT CSV:',OUT/f'surgery_candidates_{TARGET_CATEGORY}.csv'); print('ZIP:',zip_path)

# High-confidence protocol before training

Do not fine-tune an expert merely because it ranks first in one smoke test. A strong surgery target should survive:

1. larger target/control sets,
2. matched task difficulty and prompt lengths,
3. multiple prompt phrasings/seeds,
4. both `RENORMALIZE_AFTER_ABLATION=False` and `True`,
5. a real downstream metric after the cheap NLL screen,
6. optional small coalition tests.

Then compare **causally selected expert surgery** against equal-parameter random-expert and ordinary LoRA baselines.